In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import os

print(os.listdir("/kaggle/input/amazon-reviews"))

['amazon_review_polarity_csv.tgz', 'train.csv', 'test.csv']


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("/kaggle/input/amazon-reviews/train.csv")

In [ ]:
df.loc[0][2]

/tmp/ipykernel_1497/2880214942.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.loc[0][2]


"I'm reading a lot of reviews saying that this is the best 'game soundtrack' and I figured that I'd write a review to disagree a bit. This in my opinino is Yasunori Mitsuda's ultimate masterpiece. The music is timeless and I'm been listening to it for years now and its beauty simply refuses to fade.The price tag on this is pretty staggering I must say, but if you are going to buy any cd for this much money, this is the only one that I feel would be worth every penny."

In [ ]:
df.columns = ['label', 'title', 'review']

In [ ]:
df.head()

,label,title,review,text
3493923,2,Very disappointed,My son got this toy for his birthday. The kids...,Very disappointedMy son got this toy for his b...
2472516,1,Doesn't help with cats or dogs!,A person I work with highly recommended it. He...,Doesn't help with cats or dogs!A person I work...
2110064,1,Don't waste your money on this DVD,This DVD is real old school and cheezeball. Ba...,Don't waste your money on this DVDThis DVD is ...
2762210,1,EZEKIEL 13,Mr. Spong has attempted to reduce God to base ...,EZEKIEL 13Mr. Spong has attempted to reduce Go...
1438189,1,Won't fit your case.,I am the fourth review and the third to compla...,Won't fit your case.I am the fourth review and...


In [ ]:
df=df.sample(n=20000,random_state=42)

In [ ]:
df.shape

(20000, 4)

In [ ]:
df['text']=df['title']+df['review']

In [ ]:
df['text'].shape

(20000,)

In [ ]:
df[df.isnull().any(axis=1)].count()

,0
label,0
title,0
review,0
text,0


In [ ]:
df = df.dropna()

In [ ]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [ ]:
df['text'].apply(remove_html_tags)

,text
2307084,Importance of a five point harness car seat. 7...
401678,"perfectAs soon as I got it, it went on the bik..."
2984625,A little outdatedIt states in the book to have...
304692,all-time-favourite / THE novel of WW2I think I...
957202,"Best I've ever usedI had already owned one, bu..."
...,...
645189,"good watch, bad strapI've had this watch for a..."
2928608,didn't last very longI've been using this gaug...
1095739,"HMMMM....This is a descent album, conidering A..."
1516574,"Great ""origin"" superhero movie.Very good ""orig..."


In [ ]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

In [ ]:
df['text'].apply(remove_url)

,text
2307084,Importance of a five point harness car seat. 7...
401678,"perfectAs soon as I got it, it went on the bik..."
2984625,A little outdatedIt states in the book to have...
304692,all-time-favourite / THE novel of WW2I think I...
957202,"Best I've ever usedI had already owned one, bu..."
...,...
645189,"good watch, bad strapI've had this watch for a..."
2928608,didn't last very longI've been using this gaug...
1095739,"HMMMM....This is a descent album, conidering A..."
1516574,"Great ""origin"" superhero movie.Very good ""orig..."


In [ ]:
import string,time
string.punctuation
exclude = string.punctuation

In [ ]:
def remove_punc1(text):
  return text.translate(str.maketrans('','',exclude))

In [ ]:
df['text'].apply(remove_punc1)

,text
2307084,Importance of a five point harness car seat 7 ...
401678,perfectAs soon as I got it it went on the bike...
2984625,A little outdatedIt states in the book to have...
304692,alltimefavourite THE novel of WW2I think I re...
957202,Best Ive ever usedI had already owned one but ...
...,...
645189,good watch bad strapIve had this watch for abo...
2928608,didnt last very longIve been using this gauge ...
1095739,HMMMMThis is a descent album conidering ACDC m...
1516574,Great origin superhero movieVery good origin s...


In [ ]:
!pip install symspellpy

In [ ]:
from importlib.resources import files
from symspellpy import SymSpell, Verbosity

# 1. Initialize SymSpell
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = files("symspellpy").joinpath(
    "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(str(dictionary_path), term_index=0, count_index=1)

# 2. Extract valid words into a Python set for instant O(1) checks
valid_words = set(sym_spell.words.keys())


def fast_spell_check(text):
    if not isinstance(text, str):
        return ""

    tokens = text.split()
    corrected_tokens = []

    for word in tokens:
        clean_word = word.lower()
        # If the word is valid or numeric, skip SymSpell entirely
        if clean_word in valid_words or word.isnumeric():
            corrected_tokens.append(word)
        else:
            # Only run lookup on actual typos/unknown words
            suggestions = sym_spell.lookup(
                clean_word, Verbosity.CLOSEST, max_edit_distance=2, transfer_casing=False
            )
            corrected_tokens.append(
                suggestions[0].term if suggestions else word
            )

    return " ".join(corrected_tokens)


# 3. Apply to your 20k rows
df["text"] = df["text"].apply(fast_spell_check)

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# 1. Convert to a set ONCE outside the function
stop_words = set(stopwords.words('english'))

# 2. Optimized function
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word.lower() not in stop_words])

# 3. Apply to DataFrame
df['text'] = df['text'].apply(remove_stopwords)

In [ ]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stem_text(text):
    return " ".join([ps.stem(word) for word in text.split()])

df['cleaned_text'] = df['text'].apply(stem_text)

In [ ]:
df['cleaned_text'].iloc[2]

["littl outdat state book temp still air incub 102 degre bad bad high copyright 1976 info show opinion go storey guid ...' book lot detail updat inform vagu real vagu felt wast money"]

In [ ]:
df.head()

,label,title,review,text,cleaned_text
2307084,2,Importance of a five point harness car seat. 7...,"Best car seat available. Good price, easy to i...",Importance five point harness car seat 7 year ...,[import five point har car seat 7 year old sti...
401678,2,perfect,"As soon as I got it, it went on the bike and h...",perfects soon got went bike come know golding ...,[perfect soon got went bike come know gold eno...
2984625,1,A little outdated,It states in the book to have the temp. in a s...,little outdated states book temp still air inc...,[littl outdat state book temp still air incub ...
304692,2,all-time-favourite / THE novel of WW2,I think I read this book some 15 times. Still ...,all-time-favourite novel wiki think read book ...,[all-time-favourit novel wiki think read book ...
957202,2,Best I've ever used,"I had already owned one, but my buddy bought o...",Best ever used already owned one buddy bought ...,[best ever use alreadi own one buddi bought on...


In [ ]:
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt_tab")  # or nltk.download('punkt_tab') depending on nltk version

# Correct syntax:
df["cleaned_text"] = df["cleaned_text"].apply(
    lambda x: sent_tokenize(str(x)) if pd.notnull(x) else []
)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 66.4 MB/s eta 0:00:00


In [ ]:
import gensim

In [ ]:
import re
from gensim.utils import simple_preprocess

# Preprocess each sentence: strips punctuation, numbers, and lowercases automatically
corpus = [
    simple_preprocess(str(text), deacc=True) for text in df["cleaned_text"]
]

# Filter out empty token lists
corpus = [doc for doc in corpus if doc]

In [ ]:
model=gensim.models.Word2Vec(
    vector_size = 100,
window = 5,
min_count = 2,
workers = 4,
epochs = 5,
    sg=1
)

In [ ]:
model.build_vocab(df['cleaned_text'])

In [ ]:
model.train(df['cleaned_text'],total_examples=model.corpus_count,epochs=model.epochs)

(478, 125185)

In [ ]:
model.wv.most_similar("bad")

[('great stuff!', 0.24666324257850647),
 (').', 0.17826786637306213),
 ('must have!', 0.1710737943649292),
 ('!!!!!!!!!', 0.17061734199523926),
 ('!!!!!!', 0.16444021463394165),
 ('?', 0.16112184524536133),
 ('expected!!', 0.10756128281354904),
 (':)', 0.10560770332813263),
 ('save money', 0.09215974062681198),
 ('!!!!!!!!', 0.05945290997624397)]